<a href="https://colab.research.google.com/github/TarfaMajeed/Projects/blob/main/NUMBER_PLATE_DETECTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install ultralytics easyocr opencv-python matplotlib lxml


In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
from ultralytics import YOLO
import easyocr


In [ ]:
# SOURCE DATA FROM DRIVE
SRC_ROOT = "/content/drive/MyDrive/Ruqia/archive (3)"
IMG_SRC = os.path.join(SRC_ROOT, "images")
XML_SRC = os.path.join(SRC_ROOT, "annotations")

# YOLO DATASET LOCATION
YOLO_ROOT = "/content/number_plate_dataset"


In [ ]:
folders = [
    "images/train",
    "images/val",
    "labels/train",
    "labels/val"
]

for f in folders:
    os.makedirs(os.path.join(YOLO_ROOT, f), exist_ok=True)

print("YOLO folder structure created")


In [ ]:
import random
import shutil

images = os.listdir(IMG_SRC)
random.shuffle(images)

split_index = int(0.8 * len(images))
train_images = images[:split_index]
val_images = images[split_index:]

def copy_files(img_list, img_dest, xml_dest):
    for img in img_list:
        shutil.copy(os.path.join(IMG_SRC, img), img_dest)
        xml_name = img.replace(".png", ".xml").replace(".jpg", ".xml")
        shutil.copy(os.path.join(XML_SRC, xml_name), xml_dest)

copy_files(train_images,
           f"{YOLO_ROOT}/images/train",
           f"{YOLO_ROOT}/labels/train")

copy_files(val_images,
           f"{YOLO_ROOT}/images/val",
           f"{YOLO_ROOT}/labels/val")

print("Dataset split completed")


In [ ]:
CLASS_MAP = {"licence": 0}


In [ ]:
def convert_xml_to_yolo(xml_path, img_path, txt_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    img = cv2.imread(img_path)
    h, w, _ = img.shape

    with open(txt_path, "w") as f:
        for obj in root.findall("object"):
            class_name = obj.find("name").text
            class_id = CLASS_MAP[class_name]

            bbox = obj.find("bndbox")
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)

            x_center = ((xmin + xmax) / 2) / w
            y_center = ((ymin + ymax) / 2) / h
            bw = (xmax - xmin) / w
            bh = (ymax - ymin) / h

            f.write(f"{class_id} {x_center} {y_center} {bw} {bh}\n")


In [ ]:
for split in ["train", "val"]:
    label_dir = f"{YOLO_ROOT}/labels/{split}"
    img_dir = f"{YOLO_ROOT}/images/{split}"

    for file in os.listdir(label_dir):
        if file.endswith(".xml"):
            xml_path = os.path.join(label_dir, file)
            img_path = os.path.join(img_dir, file.replace(".xml", ".png"))
            txt_path = xml_path.replace(".xml", ".txt")

            convert_xml_to_yolo(xml_path, img_path, txt_path)
            os.remove(xml_path)

print("XML to YOLO conversion completed")


In [ ]:
data_yaml = """
path: /content/number_plate_dataset
train: images/train
val: images/val

nc: 1
names: ['licence']
"""

with open(f"{YOLO_ROOT}/data.yaml", "w") as f:
    f.write(data_yaml)

print("data.yaml created")


In [ ]:
model = YOLO("yolov8n.pt")

model.train(
    data=f"{YOLO_ROOT}/data.yaml",
    epochs=80,
    imgsz=640,
    batch=8,
    name="number_plate_model"
)


In [ ]:
model = YOLO("runs/detect/number_plate_model/weights/best.pt")


In [ ]:
reader = easyocr.Reader(['en'])


In [ ]:
# Folder containing all images
image_folder = "/content/drive/MyDrive/Ruqia/archive (3)/images"

plate_results = []  # to store results

for img_name in os.listdir(image_folder):

    if not img_name.lower().endswith((".jpg", ".png", ".jpeg")):
        continue

    image_path = os.path.join(image_folder, img_name)

    img = cv2.imread(image_path)
    if img is None:
        continue

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Run detection
    results = model(image_path, conf=0.3)[0]

    detected_texts = []

    for box in results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])

        # Draw bounding box
        cv2.rectangle(img_rgb, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # Crop plate
        plate_crop = img[y1:y2, x1:x2]

        if plate_crop.size == 0:
            continue

        # Preprocess for OCR
        plate_gray = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2GRAY)

        # OCR
        ocr_result = reader.readtext(plate_gray)

        for (_, text, conf) in ocr_result:
            detected_texts.append(text)

            cv2.putText(
                img_rgb,
                text,
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.9,
                (255, 0, 0),
                2
            )

    # Save result
    plate_results.append({
        "image": img_name,
        "plates": detected_texts
    })

    # Display image
    plt.figure(figsize=(10,6))
    plt.imshow(img_rgb)
    plt.title(f"Image: {img_name}")
    plt.axis("off")
    plt.show()

    print("Detected Plate(s):", detected_texts)
    print("-" * 60)
